# Tensors: PyTorch's core data structure

_PyTorch (a complete course)_

**A tensor is an n-dimensional array, like a numpy array but with GPU and autograd built in.**

A tensor is just a flat block of numbers in memory plus a small amount of bookkeeping: its shape (how many along each dimension), its dtype (what kind of number — float32, int64, bool), and its device (where the numbers live — cpu or cuda). Everything else is operations on that block.

---

This notebook is a practice scaffold for the **Tensors: PyTorch's core data structure** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)  # reproducible randn

# ---- CREATION: many ways to make a tensor ----
a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])   # from a Python list -> float32 by default
z = torch.zeros(2, 3)                 # all zeros
o = torch.ones(2, 3)                  # all ones
r = torch.arange(0, 12)               # 0,1,2,...,11  (a range, int64)
g = torch.randn(2, 3)                 # standard-normal random
I = torch.eye(3)                      # 3x3 identity matrix

print("a =\n", a)
print("arange:", r)
print("identity:\n", I)

# ---- SHAPE / DTYPE / DEVICE: the bookkeeping on every tensor ----
print("shape :", a.shape)             # torch.Size([2, 3])
print("ndim  :", a.ndim)              # 2
print("numel :", a.numel())           # 6 total elements
print("dtype :", a.dtype)             # torch.float32  (the default for floats)
print("device:", a.device)            # cpu

# ---- DTYPES: float32 default, and converting ----
i = torch.tensor([1, 2, 3])           # int literals -> int64
print("int tensor dtype :", i.dtype)  # torch.int64
f = i.to(torch.float32)               # cast to float32
print("after .to(float32):", f.dtype) # torch.float32
b = torch.tensor([1, 0, 2]).bool()    # nonzero -> True
print("bool tensor:", b)

# ---- NUMPY ROUND-TRIP + the SHARED-MEMORY gotcha ----
npy = np.array([1.0, 2.0, 3.0])       # numpy defaults to float64
t = torch.from_numpy(npy)             # SHARES the same memory buffer
print("from_numpy dtype:", t.dtype)   # torch.float64 (note: not float32!)
t[0] = 99.0                           # mutate the torch tensor...
print("numpy array is now:", npy)     # ...and the numpy array changed too: [99. 2. 3.]
safe = torch.tensor(npy)              # torch.tensor COPIES -> independent
back = a.numpy()                      # .numpy() also shares memory (CPU tensors)
print("back to numpy shape:", back.shape)

# ---- DEVICES: move to GPU if one is available ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)
a_dev = a.to(device)                  # copy onto the chosen device
print("a is on:", a_dev.device)
a_back = a_dev.cpu()                  # bring it back to CPU (e.g. before .numpy())

# ---- INDEXING / SLICING: numpy-like ----
print("first row      :", a[0])       # [1., 2., 3.]
print("second column  :", a[:, 1])    # [2., 5.]
print("top-left 2x2    :\n", a[:2, :2])
print("positive mask  :", a[a > 3])   # boolean indexing -> [4., 5., 6.]
print("single element :", a[0, 2].item())  # .item() -> plain Python float 3.0


## Visualize the data & results

_How much memory does the same 1000-element tensor take in each dtype?_

In [ ]:
import numpy as np

n = 1000  # number of elements in the tensor
dtypes = ["float64", "float32", "float16", "int8"]
bytes_per = {"float64": 8, "float32": 4, "float16": 2, "int8": 1}

# total bytes = elements * bytes-per-element (this is exactly what tensor.element_size()
# times tensor.numel() reports in PyTorch)
totals = [n * bytes_per[d] for d in dtypes]
for d, t in zip(dtypes, totals):
    print(f"{d:8s}: {t} bytes")
# float64 : 8000 bytes
# float32 : 4000 bytes
# float16 : 2000 bytes
# int8    : 1000 bytes

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Create three tensors and print each one's .shape and .dtype: (a) a 2&times;3 tensor of zeros in float32; (b) a 1-D tensor holding 0,1,2,3,4 built with torch.arange; (c) a 3&times;3 identity matrix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use the dedicated constructors instead of typing numbers by hand. — _torch.zeros, torch.arange, and torch.eye create these shapes directly._
- Pass dtype=torch.float32 to (a) and read .dtype on each. — _arange defaults to int64; only by printing dtype do you see the difference._

**Answer:** import torch
a = torch.zeros(2, 3, dtype=torch.float32)
b = torch.arange(5)            # tensor([0, 1, 2, 3, 4])
c = torch.eye(3)
for t in (a, b, c):
    print(t.shape, t.dtype)
# torch.Size([2, 3]) torch.float32
# torch.Size([5]) torch.int64
# torch.Size([3, 3]) torch.float32

</details>

**Problem 2.** Type this in Colab. Make a 1-D tensor of the numbers 0..11 with torch.arange, reshape it to 3&times;4, then to 4&times;3, then flatten it back to 1-D. Print the shape at each step.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Start from torch.arange(12). — _It gives the twelve values you will rearrange._
- Chain .reshape(3, 4), .reshape(4, 3), .reshape(-1). — _-1 tells PyTorch to infer the remaining dimension, which flattens it._

**Answer:** x = torch.arange(12)
print(x.shape)              # torch.Size([12])
x = x.reshape(3, 4)
print(x.shape)              # torch.Size([3, 4])
x = x.reshape(4, 3)
print(x.shape)              # torch.Size([4, 3])
x = x.reshape(-1)
print(x.shape)              # torch.Size([12])

</details>

**Problem 3.** Type this in Colab. Build a 4&times;4 tensor of 0..15 (use torch.arange(16).reshape(4, 4)). Then slice out: row index 1; column index 2; and the top-left 2&times;2 block. Print each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Index rows with x[1] and columns with x[:, 2]. — _The first index is the row axis; : keeps every row while picking one column._
- Slice both axes for the block: x[:2, :2]. — _:2 means indices 0 and 1 on each axis._

**Answer:** x = torch.arange(16).reshape(4, 4)
print(x[1])        # tensor([4, 5, 6, 7])
print(x[:, 2])     # tensor([ 2,  6, 10, 14])
print(x[:2, :2])   # tensor([[0, 1],
                   #         [4, 5]])

</details>

**Problem 4.** Type this in Colab. Create a column vector of shape (3, 1) holding 1,2,3 and a row vector of shape (1, 4) holding 10,20,30,40. Add them. Before you run it, predict the output shape; then verify with .shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reshape torch.tensor([1,2,3]) to (3, 1) and torch.tensor([10,20,30,40]) to (1, 4). — _Broadcasting stretches a size-1 axis to match the other operand._
- Add them; the result broadcasts to (3, 4). — _Each of the 3 rows pairs with each of the 4 columns._

**Answer:** col = torch.tensor([1, 2, 3]).reshape(3, 1)
row = torch.tensor([10, 20, 30, 40]).reshape(1, 4)
out = col + row
print(out.shape)   # torch.Size([3, 4])
print(out)
# tensor([[11, 21, 31, 41],
#         [12, 22, 32, 42],
#         [13, 23, 33, 43]])

</details>

**Problem 5.** Type this in Colab. Make a 3&times;4 tensor of random values with torch.randn(3, 4). Compute the mean of each column and the sum of each row. (Hint: reductions take a dim argument.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set a seed first with torch.manual_seed(0). — _So your numbers match a teammate's and the run is reproducible._
- Use x.mean(dim=0) for column means and x.sum(dim=1) for row sums. — _dim is the axis that collapses: dim=0 collapses rows, leaving one value per column._

**Answer:** torch.manual_seed(0)
x = torch.randn(3, 4)
print(x.mean(dim=0).shape)   # torch.Size([4])  one mean per column
print(x.sum(dim=1).shape)    # torch.Size([3])  one sum per row
print(x.mean(dim=0))
print(x.sum(dim=1))

</details>

**Problem 6.** Type this in Colab. Pick a device string ("cuda" if a GPU is available, else "cpu"). Create a float32 tensor on that device, print its .device, move it to the CPU, and print .device again.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the device string once: device = "cuda" if torch.cuda.is_available() else "cpu". — _One variable used everywhere prevents CPU/GPU mismatch errors later._
- Create with device=device, then call .to("cpu"). — _.to(...) returns a tensor on the requested device; reassign to keep it._

**Answer:** device = "cuda" if torch.cuda.is_available() else "cpu"
t = torch.ones(2, 2, dtype=torch.float32, device=device)
print(t.device)        # cuda:0 on a GPU runtime, else cpu
t = t.to("cpu")
print(t.device)        # cpu

</details>

**Problem 7.** Type this in Colab. Create a = np.array([1, 2, 3]) and t = torch.from_numpy(a). Set t[0] = 99 and print a &mdash; notice they share memory. Then make an independent copy and show that mutating it does not touch a.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mutate through the tensor, then print the numpy array. — _from_numpy wraps the same buffer, so a[0] becomes 99 too._
- Make a copy with t.clone() (or torch.tensor(a)) and mutate that instead. — _clone() allocates new memory, breaking the link to a._

**Answer:** import numpy as np
a = np.array([1, 2, 3])
t = torch.from_numpy(a)
t[0] = 99
print(a)            # [99  2  3]  -- shared memory!
indep = t.clone()
indep[1] = 0
print(a)            # [99  2  3]  -- unchanged this time

</details>

**Problem 8.** Type this in Colab. Try to create w = torch.tensor([1, 2, 3], requires_grad=True) and observe the error. Then fix it so w is a float tensor that requires gradients, and print w.requires_grad and w.dtype.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run the broken line and read the error. — _Integer tensors cannot carry gradients, so PyTorch raises a RuntimeError._
- Make the values floats: [1.0, 2.0, 3.0] (or pass dtype=torch.float32). — _Only floating-point tensors can require grad, because gradients are real-valued._

**Answer:** # Broken: RuntimeError -- only float tensors can require grad
# w = torch.tensor([1, 2, 3], requires_grad=True)

w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(w.requires_grad)   # True
print(w.dtype)           # torch.float32

</details>